# RAG Concepts, End to End: Chunking -> Embeddings -> Vector DBs -> Search -> Reranking -> MMR -> Document Loading -> Structured Output -> LCEL RAG Chain -> Memory

This notebook teaches the core building blocks of Retrieval-Augmented Generation (RAG),
one concept at a time, using the **same toy text and toy documents throughout** so you
can directly compare how each technique behaves on identical data.

**Sections (each builds on the previous):**
1. Chunking - what it is, LangChain's splitter types, code comparison, pros/cons
2. Embeddings - what they are, paid vs open-source models, code + pros/cons
3. Build-your-own embedding - step by step, plus a tiny "baby embedding" you train yourself
4. Vector databases - what they are, the major options, pros/cons
5. Pushing toy data into ChromaDB, FAISS, and Qdrant Cloud
6. Semantic search vs BM25 vs hybrid search vs metadata filtering
7. Cross-encoder reranking
8. Maximal Marginal Relevance (MMR)
9. Document loading with LangChain (PDF, text, web URL, SharePoint, AWS S3, GCS/Drive)
10. Output parsers, structured output, and Pydantic-based extraction
11. LCEL chains: wiring retrieval into a real LLM call
12. Putting it together: a minimal end-to-end RAG chain (retrieve -> MMR -> rerank -> generate)
13. Memory / conversation history with `RunnableWithMessageHistory`

Every code cell is runnable on its own once the cells above it have run. Real API calls
are made to OpenAI (embeddings + `gpt-4o-mini` chat) and Qdrant Cloud (vector storage) -
there is no mock mode.


In [1]:
# ---- Setup: env, imports, shared toy data (used by every section below) ----
import os
import sys

# Load .env from the repo root (three levels up: teaching/rag_concepts/ -> repo root)
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, os.path.join(os.getcwd(), "data"))

from dotenv import load_dotenv
load_dotenv(os.path.join(REPO_ROOT, ".env"))

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY not found - check your .env"
assert os.environ.get("QDRANT_URL"), "QDRANT_URL not found - check your .env"
assert os.environ.get("QDRANT_API_KEY"), "QDRANT_API_KEY not found - check your .env"

from toy_corpus import TOY_PARAGRAPH, TOY_DOCUMENTS, TOY_WORD, TOY_QUERY

print("Toy paragraph length (chars):", len(TOY_PARAGRAPH))
print("Toy documents:", len(TOY_DOCUMENTS))
print("Toy word:", TOY_WORD)
print("Toy query:", TOY_QUERY)


Toy paragraph length (chars): 1007
Toy documents: 8
Toy word: rainforest
Toy query: How does deforestation affect the Amazon?


## Step a) Chunking

**What is chunking?** Chunking is splitting a long document into smaller pieces
("chunks") before embedding and storing them. Embedding models have limited input
length, and retrieval works better over focused passages than one giant blob of
text - so chunking decides *what a retrieved unit of context actually looks like*.

**Kinds of chunking LangChain supports (and what each one optimizes for):**

| Splitter | How it decides where to cut | Good for |
|---|---|---|
| `CharacterTextSplitter` (fixed-length) | Cuts every N characters (with overlap) on a single separator | Simple, uniform text; fastest, least "aware" of meaning |
| `RecursiveCharacterTextSplitter` | Tries a list of separators (paragraph, sentence, word, char) in order, recursively, until chunks fit the size | General-purpose default - respects structure when possible |
| `TokenTextSplitter` | Splits by model token count instead of characters | When you must respect an embedding/LLM model's token limit exactly |
| `SemanticChunker` (langchain-experimental) | Embeds sentences and cuts where meaning shifts (embedding distance jumps) | Long-form text where topic boundaries matter more than fixed size |

We'll run all four on the **same** toy paragraph so you can see exactly how the
resulting chunks differ.


In [2]:
# ---- Step a: apply different chunking strategies to the same toy paragraph ----
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter,
)
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings

def show_chunks(name, chunks):
    print(f"--- {name}: {len(chunks)} chunks ---")
    for i, c in enumerate(chunks):
        text = c if isinstance(c, str) else c.page_content
        print(f"[{i}] ({len(text)} chars) {text!r}")
    print()

# 1. Fixed-length chunking
fixed_splitter = CharacterTextSplitter(separator=" ", chunk_size=200, chunk_overlap=20)
fixed_chunks = fixed_splitter.split_text(TOY_PARAGRAPH)
show_chunks("Fixed-length (CharacterTextSplitter, 200 chars)", fixed_chunks)

# 2. Recursive chunking
recursive_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
recursive_chunks = recursive_splitter.split_text(TOY_PARAGRAPH)
show_chunks("Recursive (RecursiveCharacterTextSplitter, 200 chars)", recursive_chunks)

# 3. Token-based chunking
token_splitter = TokenTextSplitter(chunk_size=40, chunk_overlap=5)
token_chunks = token_splitter.split_text(TOY_PARAGRAPH)
show_chunks("Token-based (TokenTextSplitter, 40 tokens)", token_chunks)

# 4. Semantic chunking (uses real OpenAI embeddings to detect topic shifts)
semantic_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
semantic_splitter = SemanticChunker(semantic_embeddings)
semantic_chunks = semantic_splitter.split_text(TOY_PARAGRAPH)
show_chunks("Semantic (SemanticChunker, OpenAI embeddings)", semantic_chunks)


/var/folders/1h/8kw0jbt134d6sgxbnf23qrhr0000gn/T/ipykernel_52674/1491313237.py:7: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


--- Fixed-length (CharacterTextSplitter, 200 chars): 6 chunks ---
[0] (195 chars) 'The Amazon rainforest is the largest tropical rainforest in the world, covering much of northwestern Brazil and extending into Colombia, Peru, and other South American countries. It is home to an'
[1] (200 chars) "It is home to an estimated 400 billion individual trees representing 16,000 species. The rainforest plays a critical role in regulating the Earth's climate by absorbing large amounts of carbon dioxide"
[2] (197 chars) 'of carbon dioxide and releasing oxygen. Scientists often call it the "lungs of the planet" because of this role, although the rainforest itself also consumes a large share of the oxygen it produces'
[3] (198 chars) "oxygen it produces through respiration and decomposition. Deforestation, driven largely by cattle ranching, logging, and agriculture, threatens the rainforest's biodiversity and its ability to store"
[4] (200 chars) 'its ability to store carbon. Indigenous communities

--- Semantic (SemanticChunker, OpenAI embeddings): 2 chunks ---
[0] (263 chars) 'The Amazon rainforest is the largest tropical rainforest in the world, covering much of northwestern Brazil and extending into Colombia, Peru, and other South American countries. It is home to an estimated 400 billion individual trees representing 16,000 species.'
[1] (743 chars) 'The rainforest plays a critical role in regulating the Earth\'s climate by absorbing large amounts of carbon dioxide and releasing oxygen. Scientists often call it the "lungs of the planet" because of this role, although the rainforest itself also consumes a large share of the oxygen it produces through respiration and decomposition. Deforestation, driven largely by cattle ranching, logging, and agriculture, threatens the rainforest\'s biodiversity and its ability to store carbon. Indigenous communities have lived in the Amazon for thousands of years and rely on the forest for food, medicine, and shelter. Conservation efforts tod

**Pros and cons**

| Strategy | Pros | Cons |
|---|---|---|
| Fixed-length | Simple, predictable, fast, no model calls | Cuts mid-sentence/mid-idea; ignores structure and meaning |
| Recursive | Respects paragraphs/sentences when possible; good general default | Still size-driven, not meaning-driven; can still split a coherent idea if it's long |
| Token-based | Exactly matches model token limits; avoids truncation errors | Token boundaries aren't sentence boundaries; needs a tokenizer |
| Semantic | Chunks align with actual topic shifts; best retrieval relevance | Slowest and costs embedding calls for every sentence; chunk sizes are unpredictable |


## Step b) Embeddings

**What is an embedding?** An embedding is a fixed-length vector of numbers that
represents the *meaning* of a piece of text. Texts with similar meaning end up with
vectors that are close together (measured by cosine similarity or distance), which is
what makes semantic search possible.

**Paid vs open-source embedding models:**

| Model | Type | Dimensions | Notes |
|---|---|---|---|
| OpenAI `text-embedding-3-small` | Paid (API) | 1536 | Strong general-purpose quality, low cost per call |
| OpenAI `text-embedding-3-large` | Paid (API) | 3072 | Higher quality, higher cost/latency |
| Cohere `embed-english-v3` | Paid (API) | 1024 | Strong for retrieval, has "search_query" vs "search_document" modes |
| `sentence-transformers/all-MiniLM-L6-v2` (HF) | Open-source, local | 384 | Fast, small, runs on CPU, good baseline |
| `BAAI/bge-large-en-v1.5` (HF) | Open-source, local | 1024 | Strong open-source retrieval quality, heavier |
| `intfloat/e5-large-v2` (HF) | Open-source, local | 1024 | Popular open-source alternative, needs "query:"/"passage:" prefixes |

We'll build an embedding function for OpenAI (real API call) and one open-source
Hugging Face model (runs locally, no key needed), then embed the **same** word and
text with both and compare.


In [3]:
# ---- Step b: embedding functions for OpenAI and an open-source HF model ----
from openai import OpenAI
from sentence_transformers import SentenceTransformer

openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
hf_model = SentenceTransformer("all-MiniLM-L6-v2")

def embed_openai(text, model="text-embedding-3-small"):
    resp = openai_client.embeddings.create(model=model, input=text)
    return resp.data[0].embedding

def embed_hf(text):
    return hf_model.encode(text).tolist()

# Embed the same toy word and toy paragraph with both models
for label, text in [("word", TOY_WORD), ("paragraph", TOY_PARAGRAPH)]:
    openai_vec = embed_openai(text)
    hf_vec = embed_hf(text)
    print(f"--- {label}: {text[:60]!r}{'...' if len(text) > 60 else ''} ---")
    print(f"OpenAI  (text-embedding-3-small): length={len(openai_vec)}  first 5 dims={openai_vec[:5]}")
    print(f"HF (all-MiniLM-L6-v2):            length={len(hf_vec)}  first 5 dims={hf_vec[:5]}")
    print()


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

--- word: 'rainforest' ---
OpenAI  (text-embedding-3-small): length=1536  first 5 dims=[0.0211944580078125, 0.00930023193359375, 0.00460052490234375, -0.008758544921875, -0.004772186279296875]
HF (all-MiniLM-L6-v2):            length=384  first 5 dims=[0.09114974737167358, 0.03192371875047684, 0.007974499836564064, 0.01382351852953434, 0.09230386465787888]



--- paragraph: 'The Amazon rainforest is the largest tropical rainforest in '... ---
OpenAI  (text-embedding-3-small): length=1536  first 5 dims=[0.0189666748046875, 0.00421905517578125, 0.049041748046875, 0.00630950927734375, 0.006214141845703125]
HF (all-MiniLM-L6-v2):            length=384  first 5 dims=[0.12321094423532486, -0.009282837621867657, 0.015830015763640404, 0.03672414273023605, 0.1494084745645523]



Notice OpenAI's vector has 1536 dimensions and the open-source MiniLM model has only
384 - different models choose different vector sizes, and you cannot mix vectors from
different models in the same similarity search (they live in different vector spaces).

**Pros and cons**

| | Pros | Cons |
|---|---|---|
| OpenAI (paid API) | High quality, no local compute/GPU needed, simple API | Costs money per call, requires network + API key, data leaves your machine |
| Open-source HF (local) | Free, runs offline, full data privacy, customizable/fine-tunable | Needs local compute (CPU/GPU), generally lower ceiling quality than best paid models, you manage versions/updates yourself |


## Step c) Building your own embedding model (step by step)

Real embedding models (OpenAI's, BGE, MiniLM, etc.) are neural networks trained on
massive text corpora with objectives like "predict the next word" or "pull similar
sentences together, push dissimilar ones apart" (contrastive learning). Here's the
process, simplified:

1. **Collect data** - large amounts of text (or paired examples of "similar"/"dissimilar" text for contrastive training).
2. **Tokenize** - break text into a fixed vocabulary of tokens/words.
3. **Pick an architecture** - a neural network (e.g. a small transformer, or even a simple co-occurrence-based method for something we can build by hand).
4. **Define a training objective** - e.g. predict a word from its neighbors (like word2vec's skip-gram), or pull matching pairs of sentences closer together.
5. **Train** - run gradient descent so the model's output vectors satisfy the objective (similar things end up close together).
6. **Freeze and use** - once trained, feed any new text through the model to get its vector; nearby vectors indicate similar meaning.

Building a *real* embedding model needs a large corpus and a GPU. Below we build a
**tiny "baby embedding"** using the same core idea as word2vec (co-occurrence
statistics), trained on our own toy corpus, entirely by hand with numpy - small
enough to run instantly, big enough to actually show the training loop and the
resulting vectors making sense.


In [4]:
# ---- Step c: a tiny "baby embedding" model trained from scratch on our toy corpus ----
import numpy as np
import re
from collections import defaultdict

# 1. Build a small training corpus out of our toy documents (lowercase words only)
corpus_text = " ".join(d["text"] for d in TOY_DOCUMENTS) + " " + TOY_PARAGRAPH
tokens = re.findall(r"[a-z']+", corpus_text.lower())
vocab = sorted(set(tokens))
word_to_id = {w: i for i, w in enumerate(vocab)}
print(f"Vocabulary size: {len(vocab)} words")

# 2. Build a co-occurrence matrix (word2vec's skip-gram intuition: words that
#    appear near each other often are related). Window size = 2.
window = 2
vocab_size = len(vocab)
cooc = np.zeros((vocab_size, vocab_size))
for i, tok in enumerate(tokens):
    center_id = word_to_id[tok]
    for j in range(max(0, i - window), min(len(tokens), i + window + 1)):
        if i == j:
            continue
        context_id = word_to_id[tokens[j]]
        cooc[center_id, context_id] += 1

# 3. "Train" a baby embedding via truncated SVD on the co-occurrence matrix -
#    this is the same linear-algebra idea behind classic embedding methods
#    like GloVe/LSA: compress a big sparse co-occurrence matrix into a small
#    dense vector per word.
EMBED_DIM = 16
U, S, Vt = np.linalg.svd(cooc, full_matrices=False)
baby_embeddings = U[:, :EMBED_DIM] * S[:EMBED_DIM]  # shape: (vocab_size, EMBED_DIM)
print(f"Baby embedding matrix shape: {baby_embeddings.shape}  (vocab_size x {EMBED_DIM})")

def baby_embed(word):
    word = word.lower()
    if word not in word_to_id:
        return None
    return baby_embeddings[word_to_id[word]]

def baby_cosine_sim(w1, w2):
    v1, v2 = baby_embed(w1), baby_embed(w2)
    if v1 is None or v2 is None:
        return None
    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))

# Sanity check: related words should be more similar than unrelated ones
print("baby vector for 'rainforest' (first 5 dims):", baby_embed("rainforest")[:5])
print("similarity(rainforest, amazon)  =", round(baby_cosine_sim("rainforest", "amazon"), 3))
print("similarity(rainforest, deforestation) =", round(baby_cosine_sim("rainforest", "deforestation"), 3))
print("similarity(rainforest, python) =", round(baby_cosine_sim("rainforest", "python"), 3))


Vocabulary size: 149 words
Baby embedding matrix shape: (149, 16)  (vocab_size x 16)
baby vector for 'rainforest' (first 5 dims): [-6.24525643 -4.3089858   1.56572437  0.07685696  1.18731493]
similarity(rainforest, amazon)  = 0.746
similarity(rainforest, deforestation) = 0.794
similarity(rainforest, python) = 0.19


Even this tiny, from-scratch model correctly learns that "rainforest" is more similar
to "amazon"/"deforestation" than to an unrelated word like "python" - purely from
co-occurrence patterns in our small toy corpus. Production embedding models scale this
same core idea (words/sentences that occur in similar contexts get similar vectors) up
to billions of tokens and much larger neural networks.


## Step d) Vector databases

**What is a vector database?** A vector database stores embeddings alongside their
original text/metadata and lets you search by *similarity* ("find the k vectors
closest to this query vector") instead of by exact match - this is what powers
semantic search and RAG retrieval.

| Vector DB | Hosting | Notes |
|---|---|---|
| **ChromaDB** | Local (embedded) or self-hosted | Zero setup, great for prototyping/demos, persists to disk |
| **FAISS** | Local (in-memory/file, library not a server) | Extremely fast, Meta's similarity-search library, no metadata filtering built in |
| **Qdrant** | Self-hosted or Qdrant Cloud (managed) | Rich metadata filtering, hybrid search support, production-ready |
| **Pinecone** | Fully managed cloud only | Very easy to scale, no self-hosting option, usage-based pricing |
| **Weaviate** | Self-hosted or managed cloud | Built-in hybrid search (BM25 + vector) and modules for many embedding providers |
| **AWS (OpenSearch/Kendra)** | Managed cloud (AWS) | Integrates with existing AWS infra; OpenSearch also does keyword search |
| **Azure AI Search** | Managed cloud (Azure) | Integrates with Azure ecosystem; strong hybrid search support |
| **GCP (Vertex AI Vector Search)** | Managed cloud (GCP) | Scales to billions of vectors; tightly coupled to GCP |

**Pros and cons**

| | Pros | Cons |
|---|---|---|
| ChromaDB | Free, trivial local setup, good for demos/small projects | Not built for massive scale or heavy concurrent production load |
| FAISS | Fastest raw similarity search, full control, free | Just a library - you build storage/metadata/persistence yourself |
| Qdrant | Rich filtering, hybrid search, can self-host or use cloud | Self-hosting adds ops work; cloud tier has usage limits/cost |
| Pinecone/Weaviate/AWS/Azure/GCP (managed) | No infra to manage, scales easily | Ongoing cost, vendor lock-in, data leaves your infra |

Below, we push our toy documents into **ChromaDB**, **FAISS**, and **Qdrant Cloud** -
all three, using real OpenAI embeddings - so you can see the same data land in three
different systems.


In [5]:
# ---- Step e: embed toy documents and push into ChromaDB, FAISS, and Qdrant Cloud ----
doc_texts = [d["text"] for d in TOY_DOCUMENTS]
doc_ids = [d["id"] for d in TOY_DOCUMENTS]
doc_metadatas = [d["metadata"] for d in TOY_DOCUMENTS]

# Embed every toy document once with OpenAI - reused by Chroma, FAISS, and Qdrant below
doc_vectors = [embed_openai(t) for t in doc_texts]
print(f"Embedded {len(doc_vectors)} documents, each with {len(doc_vectors[0])} dimensions")


Embedded 8 documents, each with 1536 dimensions


In [6]:
# --- ChromaDB (local, embedded) ---
import chromadb

chroma_client = chromadb.Client()  # in-memory for this demo
chroma_collection = chroma_client.get_or_create_collection(name="toy_docs")
chroma_collection.add(
    ids=doc_ids,
    embeddings=doc_vectors,
    documents=doc_texts,
    metadatas=doc_metadatas,
)
print("ChromaDB collection count:", chroma_collection.count())


ChromaDB collection count: 8


In [7]:
# --- FAISS (local, in-memory index) ---
import faiss

dim = len(doc_vectors[0])
faiss_index = faiss.IndexFlatIP(dim)  # inner product on normalized vectors ~ cosine similarity
faiss_vectors = np.array(doc_vectors, dtype="float32")
faiss.normalize_L2(faiss_vectors)
faiss_index.add(faiss_vectors)
faiss_id_to_doc = {i: TOY_DOCUMENTS[i] for i in range(len(TOY_DOCUMENTS))}
print("FAISS index size:", faiss_index.ntotal)


FAISS index size: 8


In [8]:
# --- Qdrant Cloud (real managed cloud instance) ---
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, PayloadSchemaType

qdrant_client = QdrantClient(url=os.environ["QDRANT_URL"], api_key=os.environ["QDRANT_API_KEY"])

QDRANT_COLLECTION = "rag_concepts_toy_docs"
qdrant_client.recreate_collection(
    collection_name=QDRANT_COLLECTION,
    vectors_config=VectorParams(size=dim, distance=Distance.COSINE),
)

# Qdrant Cloud requires a payload index before you can filter on a field (step f uses this)
qdrant_client.create_payload_index(
    collection_name=QDRANT_COLLECTION,
    field_name="topic",
    field_schema=PayloadSchemaType.KEYWORD,
)

points = [
    PointStruct(
        id=i,
        vector=doc_vectors[i],
        payload={"text": doc_texts[i], "doc_id": doc_ids[i], **doc_metadatas[i]},
    )
    for i in range(len(TOY_DOCUMENTS))
]
qdrant_client.upsert(collection_name=QDRANT_COLLECTION, points=points)
count = qdrant_client.count(collection_name=QDRANT_COLLECTION).count
print(f"Qdrant Cloud collection '{QDRANT_COLLECTION}' now has {count} points")


/var/folders/1h/8kw0jbt134d6sgxbnf23qrhr0000gn/T/ipykernel_52674/2391233704.py:8: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant_client.recreate_collection(


Qdrant Cloud collection 'rag_concepts_toy_docs' now has 8 points


## Step f) Semantic search vs BM25 vs hybrid search (+ metadata filtering)

- **Semantic search** compares the *meaning* of the query and documents using
  embeddings + vector similarity. It finds relevant results even without shared
  keywords (e.g. "car" vs "automobile").
- **BM25** is a classic keyword-based ranking algorithm (a smarter TF-IDF). It scores
  documents by how often/how rarely the query's exact terms appear. It's fast, needs
  no embeddings, and excels at exact terms (names, IDs, jargon) - but misses synonyms.
- **Hybrid search** combines both scores (e.g. weighted sum, or "reciprocal rank
  fusion") so you get semantic recall *and* exact-keyword precision.
- **Metadata filtering** narrows the candidate set by structured fields (e.g.
  `topic == "climate"`) *before or alongside* the similarity search - critical in
  production so retrieval respects things like access control, date ranges, or
  categories rather than searching the entire corpus.

We'll run all of these against the **same toy query** and the documents already
pushed to Qdrant Cloud in step e.


In [9]:
# ---- Step f: semantic search (Qdrant) ----
query_vector = embed_openai(TOY_QUERY)

semantic_hits = qdrant_client.query_points(
    collection_name=QDRANT_COLLECTION,
    query=query_vector,
    limit=4,
).points
print(f"Query: {TOY_QUERY!r}\n")
print("--- Semantic search (Qdrant, cosine similarity) ---")
for hit in semantic_hits:
    print(f"score={hit.score:.4f}  topic={hit.payload['topic']:<14}  {hit.payload['text']}")


Query: 'How does deforestation affect the Amazon?'

--- Semantic search (Qdrant, cosine similarity) ---
score=0.5952  topic=deforestation   Deforestation in the Amazon is driven mainly by cattle ranching, logging, and agricultural expansion.
score=0.5383  topic=deforestation   Satellite monitoring and reforestation projects are two key strategies used to slow Amazon deforestation.
score=0.4702  topic=climate         The Amazon rainforest absorbs large amounts of carbon dioxide and is often called the lungs of the planet.
score=0.4456  topic=climate         Climate change is accelerating the loss of biodiversity in tropical rainforests around the world.


In [10]:
# ---- Step f: BM25 (keyword-based, local, no embeddings/API calls) ----
from rank_bm25 import BM25Okapi

def tokenize(text):
    return re.findall(r"[a-z0-9']+", text.lower())

tokenized_docs = [tokenize(d["text"]) for d in TOY_DOCUMENTS]
bm25 = BM25Okapi(tokenized_docs)

bm25_scores = bm25.get_scores(tokenize(TOY_QUERY))
bm25_ranked = sorted(zip(TOY_DOCUMENTS, bm25_scores), key=lambda x: x[1], reverse=True)

print("--- BM25 search (keyword overlap) ---")
for doc, score in bm25_ranked[:4]:
    print(f"score={score:.4f}  topic={doc['metadata']['topic']:<14}  {doc['text']}")


--- BM25 search (keyword overlap) ---
score=1.0058  topic=deforestation   Deforestation in the Amazon is driven mainly by cattle ranching, logging, and agricultural expansion.
score=1.0058  topic=deforestation   Satellite monitoring and reforestation projects are two key strategies used to slow Amazon deforestation.
score=0.0000  topic=climate         The Amazon rainforest absorbs large amounts of carbon dioxide and is often called the lungs of the planet.
score=0.0000  topic=culture         Indigenous communities have lived in the Amazon rainforest for thousands of years, relying on it for food and medicine.


In [11]:
# ---- Step f: hybrid search (reciprocal rank fusion of semantic + BM25) ----
def reciprocal_rank_fusion(rankings, k=60):
    # rankings: list of ranked-doc-id lists (best first); fuse into one score per doc
    scores = defaultdict(float)
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking):
            scores[doc_id] += 1.0 / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

semantic_ranking = [hit.payload["doc_id"] for hit in semantic_hits]
bm25_ranking = [doc["id"] for doc, _ in bm25_ranked]

fused = reciprocal_rank_fusion([semantic_ranking, bm25_ranking])
doc_by_id = {d["id"]: d for d in TOY_DOCUMENTS}

print("--- Hybrid search (Reciprocal Rank Fusion of semantic + BM25) ---")
for doc_id, score in fused[:4]:
    doc = doc_by_id[doc_id]
    print(f"fused_score={score:.4f}  topic={doc['metadata']['topic']:<14}  {doc['text']}")


--- Hybrid search (Reciprocal Rank Fusion of semantic + BM25) ---
fused_score=0.0328  topic=deforestation   Deforestation in the Amazon is driven mainly by cattle ranching, logging, and agricultural expansion.
fused_score=0.0323  topic=deforestation   Satellite monitoring and reforestation projects are two key strategies used to slow Amazon deforestation.
fused_score=0.0317  topic=climate         The Amazon rainforest absorbs large amounts of carbon dioxide and is often called the lungs of the planet.
fused_score=0.0303  topic=climate         Climate change is accelerating the loss of biodiversity in tropical rainforests around the world.


In [12]:
# ---- Step f: metadata filtering (narrow the search to one topic) ----
from qdrant_client.models import Filter, FieldCondition, MatchValue

filtered_hits = qdrant_client.query_points(
    collection_name=QDRANT_COLLECTION,
    query=query_vector,
    query_filter=Filter(
        must=[FieldCondition(key="topic", match=MatchValue(value="deforestation"))]
    ),
    limit=4,
).points

print("--- Semantic search + metadata filter (topic == 'deforestation') ---")
for hit in filtered_hits:
    print(f"score={hit.score:.4f}  topic={hit.payload['topic']:<14}  {hit.payload['text']}")


--- Semantic search + metadata filter (topic == 'deforestation') ---
score=0.5952  topic=deforestation   Deforestation in the Amazon is driven mainly by cattle ranching, logging, and agricultural expansion.
score=0.5383  topic=deforestation   Satellite monitoring and reforestation projects are two key strategies used to slow Amazon deforestation.


Compare the four result sets above: semantic search pulls in the "climate" doc even
though it doesn't share exact words with the query; BM25 favors documents with the
literal word "deforestation"; hybrid search blends both rankings; and metadata
filtering forces the search to only consider `topic == "deforestation"` regardless of
score. In production RAG, hybrid search + metadata filtering together are usually what
you want - pure semantic search alone often under- or over-retrieves.


## Step g) Cross-encoder reranking

Vector search (a "bi-encoder": query and document are embedded *separately*, then
compared by similarity) is fast but approximate. A **cross-encoder** reranker instead
feeds the query and each candidate document **together** into one model, which
directly scores how relevant that pair is - much more accurate, but too slow to run
over an entire database. The standard pattern: use vector search to fetch a handful of
candidates fast, then use a cross-encoder to **rerank** just those candidates for
final precision.


In [13]:
# ---- Step g: rerank the semantic-search candidates with a cross-encoder ----
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

candidates = [hit.payload["text"] for hit in semantic_hits]
pairs = [(TOY_QUERY, doc_text) for doc_text in candidates]
rerank_scores = reranker.predict(pairs)

reranked = sorted(zip(candidates, rerank_scores), key=lambda x: x[1], reverse=True)

print(f"Query: {TOY_QUERY!r}\n")
print("--- Before reranking (semantic search order) ---")
for doc_text in candidates:
    print(f"  {doc_text}")

print("\n--- After cross-encoder reranking ---")
for doc_text, score in reranked:
    print(f"score={score:.4f}  {doc_text}")


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Query: 'How does deforestation affect the Amazon?'

--- Before reranking (semantic search order) ---
  Deforestation in the Amazon is driven mainly by cattle ranching, logging, and agricultural expansion.
  Satellite monitoring and reforestation projects are two key strategies used to slow Amazon deforestation.
  The Amazon rainforest absorbs large amounts of carbon dioxide and is often called the lungs of the planet.
  Climate change is accelerating the loss of biodiversity in tropical rainforests around the world.

--- After cross-encoder reranking ---
score=7.1337  Deforestation in the Amazon is driven mainly by cattle ranching, logging, and agricultural expansion.
score=1.7540  Satellite monitoring and reforestation projects are two key strategies used to slow Amazon deforestation.
score=-1.8072  The Amazon rainforest absorbs large amounts of carbon dioxide and is often called the lungs of the planet.
score=-7.7566  Climate change is accelerating the loss of biodiversity in tropica

The cross-encoder often reorders the candidates compared to the raw vector-similarity
ranking, because it directly reads the query and document together instead of
comparing two independently-computed vectors.


## Step h) Maximal Marginal Relevance (MMR)

Plain top-k similarity search can return several near-duplicate documents (all
relevant, but redundant). **MMR** re-selects results to balance two goals:
**relevance** to the query *and* **diversity** from documents already selected. It
iteratively picks the next document that maximizes:

```
MMR = lambda * sim(doc, query) - (1 - lambda) * max(sim(doc, already_selected))
```

`lambda` close to 1 -> mostly relevance (like plain top-k). `lambda` close to 0 ->
mostly diversity. We'll implement this directly and apply it to our toy documents.


In [14]:
# ---- Step h: Maximal Marginal Relevance, implemented from scratch ----
def cosine_sim(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def mmr(query_vector, doc_vectors, doc_texts, k=4, lambda_param=0.5):
    selected_idx = []
    candidate_idx = list(range(len(doc_vectors)))

    while len(selected_idx) < k and candidate_idx:
        best_score, best_i = -float("inf"), None
        for i in candidate_idx:
            relevance = cosine_sim(query_vector, doc_vectors[i])
            diversity_penalty = (
                max(cosine_sim(doc_vectors[i], doc_vectors[j]) for j in selected_idx)
                if selected_idx else 0.0
            )
            score = lambda_param * relevance - (1 - lambda_param) * diversity_penalty
            if score > best_score:
                best_score, best_i = score, i
        selected_idx.append(best_i)
        candidate_idx.remove(best_i)

    return [(doc_texts[i], doc_vectors[i]) for i in selected_idx]

# Plain top-k by similarity (for comparison)
sims = [(TOY_DOCUMENTS[i]["text"], cosine_sim(query_vector, doc_vectors[i])) for i in range(len(doc_vectors))]
top_k_plain = sorted(sims, key=lambda x: x[1], reverse=True)[:4]

print(f"Query: {TOY_QUERY!r}\n")
print("--- Plain top-k (similarity only, no diversity) ---")
for text, score in top_k_plain:
    print(f"score={score:.4f}  {text}")

mmr_results = mmr(query_vector, doc_vectors, doc_texts, k=4, lambda_param=0.5)
print("\n--- MMR (lambda=0.5: balance relevance and diversity) ---")
for text, _ in mmr_results:
    print(f"  {text}")


Query: 'How does deforestation affect the Amazon?'

--- Plain top-k (similarity only, no diversity) ---
score=0.5952  Deforestation in the Amazon is driven mainly by cattle ranching, logging, and agricultural expansion.
score=0.5383  Satellite monitoring and reforestation projects are two key strategies used to slow Amazon deforestation.
score=0.4702  The Amazon rainforest absorbs large amounts of carbon dioxide and is often called the lungs of the planet.
score=0.4456  Climate change is accelerating the loss of biodiversity in tropical rainforests around the world.

--- MMR (lambda=0.5: balance relevance and diversity) ---
  Deforestation in the Amazon is driven mainly by cattle ranching, logging, and agricultural expansion.
  The Amazon rainforest absorbs large amounts of carbon dioxide and is often called the lungs of the planet.
  Satellite monitoring and reforestation projects are two key strategies used to slow Amazon deforestation.
  Indigenous communities have lived in the Amaz

With plain top-k, the two most similar "deforestation" documents both get selected,
crowding out other relevant-but-different angles (climate, culture). MMR's diversity
penalty pushes at least one of those near-duplicates out in favor of a document that's
still relevant to the query but covers new ground - which is exactly what you want when
assembling context for an LLM: relevant *and* non-redundant.


## Step i) Document loading with LangChain

Everything so far (chunking, embeddings, vector DBs, search) assumed the text was
already a Python string. In a real pipeline, that text has to come *from* somewhere -
a PDF report, a web page, a plain text file, a SharePoint site, an S3 bucket, a GCS
bucket or Google Drive. LangChain's **document loaders** are the layer that turns all
of these different sources into a common `Document` object (`.page_content` +
`.metadata`), so every downstream step (chunking, embedding, etc.) doesn't need to
know or care where the text originally came from.

**What LangChain document loading supports (partial list - there are 150+ loaders):**

| Source | Loader class | Runs in this notebook? |
|---|---|---|
| Local PDF | `PyPDFLoader` (also `PyMuPDFLoader`, `UnstructuredPDFLoader`) | Yes - real toy PDF |
| Local plain text | `TextLoader` | Yes - real toy `.txt` file |
| Web page / URL | `WebBaseLoader` (also `UnstructuredURLLoader`) | Yes - real public URL |
| Microsoft SharePoint | `SharePointLoader` | Code shown, not executed (needs Azure AD app + tenant credentials) |
| AWS S3 | `S3FileLoader` / `S3DirectoryLoader` | Code shown, not executed (needs AWS credentials + a real bucket) |
| Google Cloud Storage | `GCSFileLoader` / `GCSDirectoryLoader` | Code shown, not executed (needs a GCP service account + bucket) |
| Google Drive | `GoogleDriveLoader` | Code shown, not executed (needs Google OAuth credentials) |
| Notion | `NotionDirectoryLoader` | Not shown (same idea as SharePoint/GDrive: needs an API token) |
| Confluence | `ConfluenceLoader` | Not shown (same idea: needs Confluence API credentials) |
| CSV | `CSVLoader` | Not shown (same idea as `TextLoader`, one `Document` per row) |
| Directory of mixed files | `DirectoryLoader` | Not shown (wraps any of the above loaders over every file in a folder) |

The pattern is the same everywhere: **install the integration package -> instantiate
the loader with a source (path/URL/bucket/credentials) -> call `.load()` -> get back
a list of `Document` objects** ready for chunking (step a) and embedding (step b).


In [15]:
# ---- Step i: real loaders - PDF, plain text, and a web URL ----
os.environ.setdefault("USER_AGENT", "rag-concepts-teaching-demo/1.0")
from langchain_community.document_loaders import PyPDFLoader, TextLoader, WebBaseLoader

# 1. PDF loader - loads data/toy_document.pdf (generated for this demo)
pdf_docs = PyPDFLoader("data/toy_document.pdf").load()
print(f"--- PyPDFLoader: {len(pdf_docs)} page(s) ---")
print(f"metadata: {pdf_docs[0].metadata}")
print(f"content preview: {pdf_docs[0].page_content[:200]!r}\n")

# 2. Plain text loader - loads data/toy_document.txt
text_docs = TextLoader("data/toy_document.txt").load()
print(f"--- TextLoader: {len(text_docs)} document(s) ---")
print(f"metadata: {text_docs[0].metadata}")
print(f"content preview: {text_docs[0].page_content[:200]!r}\n")

# 3. Web loader - loads a real, stable public URL
web_docs = WebBaseLoader("https://en.wikipedia.org/wiki/Amazon_rainforest").load()
print(f"--- WebBaseLoader: {len(web_docs)} document(s) ---")
print(f"metadata: {web_docs[0].metadata}")
print(f"content preview: {web_docs[0].page_content[:200]!r}")


--- PyPDFLoader: 1 page(s) ---
metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-07-18T23:19:29+09:00', 'author': 'anonymous', 'keywords': '', 'moddate': '2026-07-18T23:19:29+09:00', 'subject': 'unspecified', 'title': 'untitled', 'trapped': '/False', 'source': 'data/toy_document.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}
content preview: 'Toy Document for LangChain Document Loading Demo\nThis is a small toy PDF used to demonstrate PyPDFLoader.\nIt talks about the Amazon rainforest, the same topic used\nthroughout this notebook so retrieva'

--- TextLoader: 1 document(s) ---
metadata: {'source': 'data/toy_document.txt'}
content preview: 'Toy Document for LangChain Document Loading Demo\n\nThis is a small toy plain-text file used to demonstrate TextLoader.\nIt talks about the Amazon rainforest, the same topic used throughout\nthis notebook'



--- WebBaseLoader: 1 document(s) ---
metadata: {'source': 'https://en.wikipedia.org/wiki/Amazon_rainforest', 'title': 'Amazon rainforest - Wikipedia', 'language': 'en'}
content preview: '\n\n\n\nAmazon rainforest - Wikipedia\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nJump to content\n\n\n\n\n\n\n\nMain menu\n\n\n\n\n\nMain menu\nmove to sidebar\nhide\n\n\n\n\t\tNavigation\n\t\n\n\nMain pageContentsCurrent eventsRandom articleAbo'


All three return the same `Document` shape (`.page_content`, `.metadata`) regardless of
the very different sources they came from - a PDF page, a local file, and a live web
page. That's the whole point of the loader abstraction: everything downstream (step a's
chunkers, step b's embedders) works identically no matter which loader produced the
`Document`.

The cloud-storage loaders below (SharePoint, S3, GCS/Google Drive) follow the exact
same `.load()` pattern - shown here as real, runnable code, but **not executed** in
this notebook since they require real cloud credentials (an Azure AD app registration,
AWS IAM credentials, or a GCP service account) that aren't part of this demo's `.env`.


In [16]:
# ---- Step i: cloud document loaders - real code, NOT executed (no cloud creds in this demo) ----
# `if False:` keeps this syntactically real, runnable code without making network calls -
# flip to `if True:` and supply real credentials in your own .env to actually run these.

if False:
    # --- Microsoft SharePoint ---
    from langchain_community.document_loaders.sharepoint import SharePointLoader

    sharepoint_loader = SharePointLoader(
        document_library_id="<your-sharepoint-document-library-id>",
        client_id="<azure-ad-app-client-id>",
        client_secret="<azure-ad-app-client-secret>",  # or use certificate-based auth
        tenant_id="<azure-ad-tenant-id>",
    )
    sharepoint_docs = sharepoint_loader.load()

    # --- AWS S3 ---
    from langchain_community.document_loaders import S3FileLoader, S3DirectoryLoader

    s3_file_docs = S3FileLoader(bucket="<your-bucket>", key="<path/to/file.pdf>").load()
    s3_dir_docs = S3DirectoryLoader(bucket="<your-bucket>", prefix="<path/prefix/>").load()
    # Both use boto3 under the hood and read AWS credentials the standard way
    # (AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY env vars, or an IAM role).

    # --- Google Cloud Storage ---
    from langchain_google_community import GCSFileLoader, GCSDirectoryLoader

    gcs_file_docs = GCSFileLoader(project_name="<gcp-project>", bucket="<your-bucket>", blob="<path/to/file.pdf>").load()
    gcs_dir_docs = GCSDirectoryLoader(project_name="<gcp-project>", bucket="<your-bucket>").load()
    # Reads credentials via GOOGLE_APPLICATION_CREDENTIALS (a service-account JSON key).

    # --- Google Drive ---
    from langchain_google_community import GoogleDriveLoader

    gdrive_docs = GoogleDriveLoader(
        folder_id="<google-drive-folder-id>",
        credentials_path="<path/to/oauth-credentials.json>",
    ).load()

print("Cloud loader code above is real (not pseudocode) but intentionally skipped -")
print("no SharePoint/AWS/GCP credentials are configured for this teaching demo.")


Cloud loader code above is real (not pseudocode) but intentionally skipped -
no SharePoint/AWS/GCP credentials are configured for this teaching demo.


**Pros and cons of each document loading approach**

| Source | Pros | Cons |
|---|---|---|
| Local PDF/text | Zero external dependencies, fast, fully offline | You must already have the file locally; PDF text extraction can mangle complex layouts/tables |
| Web URL | No file management, always gets the latest version | Fragile (page structure changes break scraping), needs network access, may hit paywalls/robots.txt restrictions |
| SharePoint | Direct access to enterprise docs where many companies already store them | Needs Azure AD app registration and admin consent - real setup overhead |
| AWS S3 | Scales to huge document sets, integrates with existing AWS pipelines | Requires AWS credentials/IAM permissions; another cloud bill |
| Google Cloud Storage / Drive | Same as S3 for GCS; Drive loader is great for non-technical teams already using Drive | Requires GCP service account or OAuth setup; Drive API has usage quotas |


## Step j) Output parsers, structured output, and Pydantic-based extraction

**What it is:** By default, an LLM returns free-form text. An **output parser**
converts that text into a structured Python object (a dict, a list, or - most
usefully - a **Pydantic model** with typed, validated fields). "Structured output"
means asking the LLM to return data that fits a schema you define, instead of prose
you'd have to regex/parse yourself.

**Why it's used:** Free-form LLM text is unreliable to parse downstream (formats
drift, fields get renamed, numbers get spelled out). Structured output makes the
LLM's response **type-safe and predictable** - your code can trust `result.price`
is a `float`, `result.tags` is a `list[str]`, etc., the same way you'd trust an API
response validated against a schema.

**When it's used:**
- Extracting fields from unstructured text (invoices, resumes, support tickets)
- Classification (mapping free text to a fixed set of labels/categories)
- Populating a database row or API request body from natural language
- Tool/function-calling arguments (the LLM must produce arguments matching a tool's schema)
- Turning a document (like the ones loaded in step i) into structured metadata for RAG filtering

**Multiple use cases where this is the best fit:**

| Use case | Why structured output wins here |
|---|---|
| Resume parsing | Need fields like `name`, `email`, `years_experience` as real typed data, not paragraphs |
| Customer support ticket triage | Classify into a fixed `category` enum + extract `urgency`, `sentiment` reliably |
| Invoice/receipt extraction | Need exact `total`, `date`, `vendor` fields to feed into accounting software |
| Populating metadata for RAG filtering | Turn a raw document (step i) into `{topic, date, author}` fields usable in Qdrant's metadata filter (step f) |
| Agent tool-calling | The LLM's "which tool, which arguments" decision must be a valid structured call, not free text |

We'll extract structured fields from a toy sentence about the Amazon rainforest using
`ChatOpenAI.with_structured_output()` with a Pydantic model.


In [17]:
# ---- Step j: Pydantic-based structured output extraction (real OpenAI call) ----
from typing import Literal
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

chat_llm = ChatOpenAI(model="gpt-4o-mini", api_key=os.environ["OPENAI_API_KEY"], temperature=0)

class DocumentInsight(BaseModel):
    """Structured facts extracted from a passage of text."""
    main_topic: str = Field(description="The single main subject of the passage, in 1-3 words")
    sentiment: Literal["positive", "neutral", "negative"] = Field(description="Overall tone toward the topic")
    key_entities: list[str] = Field(description="Named entities (places, organizations, species) mentioned")
    mentions_deforestation: bool = Field(description="Whether deforestation/forest loss is discussed")

structured_llm = chat_llm.with_structured_output(DocumentInsight)

# Reuse a toy document from earlier (step e's TOY_DOCUMENTS) as the extraction input
sample_text = TOY_DOCUMENTS[1]["text"]  # the deforestation-topic document
insight = structured_llm.invoke(f"Extract structured insight from this passage:\n\n{sample_text}")

print(f"Input text: {sample_text!r}\n")
print("Extracted structured output (a real DocumentInsight Pydantic object):")
print(f"  type: {type(insight)}")
print(f"  main_topic: {insight.main_topic!r}")
print(f"  sentiment: {insight.sentiment!r}")
print(f"  key_entities: {insight.key_entities}")
print(f"  mentions_deforestation: {insight.mentions_deforestation}")
print(f"\nAs a dict (e.g. ready to store as Qdrant metadata): {insight.model_dump()}")


Input text: 'Deforestation in the Amazon is driven mainly by cattle ranching, logging, and agricultural expansion.'

Extracted structured output (a real DocumentInsight Pydantic object):
  type: <class '__main__.DocumentInsight'>
  main_topic: 'Deforestation'
  sentiment: 'negative'
  key_entities: ['Amazon']
  mentions_deforestation: True

As a dict (e.g. ready to store as Qdrant metadata): {'main_topic': 'Deforestation', 'sentiment': 'negative', 'key_entities': ['Amazon'], 'mentions_deforestation': True}


Note `insight` is a real, validated `DocumentInsight` instance - not a string you'd
need to `json.loads()` and hope for the best. If the LLM's output doesn't match the
schema, Pydantic validation raises an error immediately instead of silently passing
bad data downstream. This same pattern (`with_structured_output` + a Pydantic model)
is exactly what powers LangChain's tool-calling and agent argument parsing under the
hood.


## Step k) LCEL chains: wiring retrieval into an LLM call

Everything through step j used single, standalone calls (one embedding call, one
search, one structured-extraction call). **LCEL** (LangChain Expression Language) is
how you compose several of these steps into one pipeline using the `|` (pipe)
operator - the output of each piece becomes the input of the next, just like Unix
pipes.

A typical RAG chain looks like:

```text
question --> [retriever] --> context  \
                                        --> [prompt template] --> [LLM] --> [output parser] --> answer
question ------------------------------/
```

Every piece (`retriever`, `prompt`, `llm`, `parser`) is a **Runnable** - they all
implement the same `.invoke()` interface, which is exactly why they can be chained
with `|` regardless of what each one internally does (a vector search vs an HTTP call
to OpenAI vs simple string formatting).

We'll build this chain using the retriever function from step f (Qdrant semantic
search) and wire it into a real prompt + real `gpt-4o-mini` call.


In [18]:
# ---- Step k: an LCEL retrieve -> prompt -> generate chain ----
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

def retrieve_from_qdrant(question: str, k: int = 3) -> str:
    """Runnable-compatible retriever: embeds the question, searches Qdrant (step e's
    collection), and returns the top-k document texts joined as one context string."""
    q_vector = embed_openai(question)
    hits = qdrant_client.query_points(collection_name=QDRANT_COLLECTION, query=q_vector, limit=k).points
    return "\n".join(f"- {hit.payload['text']}" for hit in hits)

retriever_runnable = RunnableLambda(retrieve_from_qdrant)

rag_prompt = ChatPromptTemplate.from_template(
    "You are answering questions using retrieved context. Synthesize a helpful answer by "
    "connecting the context to the question, even if the context describes causes, related "
    "facts, or partial information rather than stating the answer outright. Only say the "
    "context is insufficient if it is truly unrelated to the question.\n\n"
    "Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"
)

# The LCEL chain: question flows into both branches of RunnablePassthrough.assign,
# retriever_runnable fills "context", then prompt -> llm -> string parser.
lcel_chain = (
    RunnablePassthrough.assign(context=lambda x: retrieve_from_qdrant(x["question"]))
    | rag_prompt
    | chat_llm
    | StrOutputParser()
)

answer = lcel_chain.invoke({"question": TOY_QUERY})
print(f"Question: {TOY_QUERY!r}\n")
print("Retrieved context:")
print(retrieve_from_qdrant(TOY_QUERY))
print(f"\nLCEL chain answer:\n{answer}")


Question: 'How does deforestation affect the Amazon?'

Retrieved context:


- Deforestation in the Amazon is driven mainly by cattle ranching, logging, and agricultural expansion.
- Satellite monitoring and reforestation projects are two key strategies used to slow Amazon deforestation.
- The Amazon rainforest absorbs large amounts of carbon dioxide and is often called the lungs of the planet.

LCEL chain answer:
Deforestation in the Amazon has significant impacts on the environment and climate. As the rainforest is often referred to as the "lungs of the planet," it plays a crucial role in absorbing carbon dioxide, which helps regulate the Earth's atmosphere. When trees are cut down for cattle ranching, logging, and agricultural expansion, this carbon storage capacity is diminished, leading to increased levels of carbon dioxide in the atmosphere. Additionally, the loss of trees disrupts local ecosystems, threatens biodiversity, and can alter weather patterns. Efforts like satellite monitoring and reforestation projects are essential in addressing these issues 

This is the missing piece from every earlier search section: steps f-h showed you
*which documents* get retrieved and in what order, but never turned that into an
actual answer. The LCEL chain above is the smallest possible version of what every
production RAG system does: retrieve context, template it into a prompt, call an LLM,
parse the output - each step a plain `Runnable`, composed with `|`.


## Step l) Putting it together: a minimal end-to-end RAG chain

Step k's chain used plain semantic search for retrieval. Here we assemble a more
complete pipeline that reuses **every retrieval technique already built** in this
notebook:

```text
question
   |
   v
[1] embed query (step b)
   |
   v
[2] semantic search in Qdrant, fetch extra candidates (step f)
   |
   v
[3] MMR re-selection for relevance + diversity (step h)
   |
   v
[4] cross-encoder reranking for final precision (step g)
   |
   v
[5] prompt template with the reranked context (step k's pattern)
   |
   v
[6] gpt-4o-mini generates the final answer
```

This is the capstone: chunking/embeddings/vector-DB/search/MMR/reranking are no
longer separate demonstrations - they're stages of one real, runnable RAG pipeline
that ends in an actual generated answer.


In [19]:
# ---- Step l: full RAG pipeline - retrieve -> MMR -> rerank -> generate ----
def full_rag_pipeline(question: str, fetch_k: int = 6, mmr_k: int = 4, final_k: int = 3) -> dict:
    # [1] + [2] embed query, over-fetch candidates from Qdrant
    q_vector = embed_openai(question)
    hits = qdrant_client.query_points(collection_name=QDRANT_COLLECTION, query=q_vector, limit=fetch_k).points
    candidate_texts = [hit.payload["text"] for hit in hits]
    candidate_vectors = [embed_openai(t) for t in candidate_texts]  # re-embed for MMR's vector math

    # [3] MMR re-selection (reuses the mmr() function defined in step h)
    mmr_selected = mmr(q_vector, candidate_vectors, candidate_texts, k=mmr_k, lambda_param=0.5)
    mmr_texts = [text for text, _ in mmr_selected]

    # [4] cross-encoder reranking (reuses the reranker loaded in step g)
    pairs = [(question, text) for text in mmr_texts]
    scores = reranker.predict(pairs)
    reranked_texts = [t for t, _ in sorted(zip(mmr_texts, scores), key=lambda x: x[1], reverse=True)][:final_k]

    # [5] + [6] prompt template + real LLM generation
    context = "\n".join(f"- {t}" for t in reranked_texts)
    prompt = rag_prompt.format(context=context, question=question)
    response = chat_llm.invoke(prompt)

    return {"context_used": reranked_texts, "answer": response.content}

result = full_rag_pipeline(TOY_QUERY)
print(f"Question: {TOY_QUERY!r}\n")
print("Final context after semantic search -> MMR -> reranking:")
for t in result["context_used"]:
    print(f"  - {t}")
print(f"\nGenerated answer:\n{result['answer']}")


Question: 'How does deforestation affect the Amazon?'

Final context after semantic search -> MMR -> reranking:
  - Deforestation in the Amazon is driven mainly by cattle ranching, logging, and agricultural expansion.
  - Satellite monitoring and reforestation projects are two key strategies used to slow Amazon deforestation.
  - The Amazon rainforest absorbs large amounts of carbon dioxide and is often called the lungs of the planet.

Generated answer:
Deforestation in the Amazon has significant impacts on the environment, primarily because the rainforest plays a crucial role in absorbing carbon dioxide, which helps mitigate climate change. As trees are cut down for cattle ranching, logging, and agricultural expansion, this vital carbon sink is diminished, leading to increased levels of carbon dioxide in the atmosphere. Additionally, the loss of trees disrupts local ecosystems, threatens biodiversity, and can alter weather patterns. Efforts like satellite monitoring and reforestation 

Notice `full_rag_pipeline` doesn't introduce a single new retrieval idea - every
building block (`embed_openai`, `qdrant_client.query_points`, `mmr`, `reranker`,
`rag_prompt`, `chat_llm`) was already defined in an earlier step. This is the whole
point of building each piece separately first: once each one works and is understood,
composing them into a real pipeline is just calling them in sequence.


## Step m) Memory / conversation history

Every chain so far is **stateless**: each `.invoke()` call knows nothing about
previous calls. Real chat applications need **memory** - the model has to remember
what was said earlier in the conversation (e.g. "what did I just ask you?" or a
follow-up question that uses "it"/"that" to refer to something mentioned before).

**How LangChain helps with this:**
- **Modern approach - `RunnableWithMessageHistory`**: wraps *any* LCEL runnable
  (including our RAG chain from step k/l) with automatic history management. You give
  it a `session_id`, and it stores/replays that session's messages automatically -
  the underlying chain doesn't need to know memory exists.
- **Legacy approach - `ConversationBufferMemory`** (older API, still functional but
  LangChain has moved away from it in favor of the runnable-based approach above):
  manually attached memory objects that injected chat history into the prompt. Worth
  recognizing in older tutorials/codebases, but `RunnableWithMessageHistory` is the
  current idiomatic path.
- Under the hood, history is just a list of messages (`HumanMessage`/`AIMessage`)
  stored in a `BaseChatMessageHistory` implementation - in-memory for a demo, or a
  real backing store (Redis, Postgres, DynamoDB) in production.

**Note:** running the code below prints a real `LangChainDeprecationWarning` -
LangChain now points newer projects toward LangGraph's built-in persistence
(`langgraph.checkpoint`) as the longer-term replacement for `RunnableWithMessageHistory`.
It's shown here because it's still the simplest, most direct way to add memory to a
plain LCEL chain today, and you'll see it throughout current tutorials/codebases - just
know that LangGraph is where this is headed for new, more complex agent projects.

We'll wrap step k's LCEL chain with `RunnableWithMessageHistory` and show a real
follow-up question that only makes sense *with* memory of the first turn.


In [20]:
# ---- Step m: conversation memory with RunnableWithMessageHistory ----
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory

conversational_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant answering questions about the Amazon rainforest."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])
conversational_chain = conversational_prompt | chat_llm | StrOutputParser()

# In-memory store of chat histories, keyed by session_id (swap for Redis/Postgres in production)
session_store: dict[str, BaseChatMessageHistory] = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in session_store:
        session_store[session_id] = InMemoryChatMessageHistory()
    return session_store[session_id]

chain_with_memory = RunnableWithMessageHistory(
    conversational_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

config = {"configurable": {"session_id": "toy-session-1"}}

turn1 = chain_with_memory.invoke({"input": "What is the Amazon rainforest and where is it located?"}, config=config)
print("Turn 1 - 'What is the Amazon rainforest and where is it located?'")
print(turn1)

# This follow-up only makes sense WITH memory of turn 1 ("it" = the Amazon rainforest)
turn2 = chain_with_memory.invoke({"input": "What is the biggest threat to it?"}, config=config)
print("\nTurn 2 - 'What is the biggest threat to it?' (relies on memory of turn 1)")
print(turn2)

print(f"\nMessages stored for this session: {len(get_session_history('toy-session-1').messages)}")


/Users/utsabchakraborty/Documents/Edureka_Full_Course/Live_Class_Codes/Coding_Agent_Enabled_Demo/teaching/rag_concepts/venv/lib/python3.14/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Turn 1 - 'What is the Amazon rainforest and where is it located?'
The Amazon rainforest is a vast tropical rainforest located in South America, primarily within Brazil, but it also extends into several other countries, including Peru, Colombia, Venezuela, Ecuador, Bolivia, Guyana, Suriname, and French Guiana. It is the largest rainforest in the world, covering approximately 5.5 million square kilometers (2.1 million square miles).

The Amazon rainforest is known for its incredible biodiversity, housing millions of species of plants, animals, and microorganisms, many of which are not found anywhere else on Earth. It plays a crucial role in regulating the global climate, producing significant amounts of oxygen, and acting as a carbon sink, which helps mitigate climate change. The Amazon River, one of the longest rivers in the world, flows through this rainforest, further contributing to its unique ecosystem.



Turn 2 - 'What is the biggest threat to it?' (relies on memory of turn 1)
The biggest threats to the Amazon rainforest include:

1. **Deforestation**: This is the most significant threat, primarily driven by agricultural expansion, logging, and infrastructure development. Large areas of the forest are cleared for cattle ranching, soy cultivation, and palm oil plantations.

2. **Illegal Logging**: Unsustainable and illegal logging practices contribute to forest degradation and loss of biodiversity. This often occurs in protected areas and indigenous lands.

3. **Mining**: The extraction of minerals and resources, such as gold and oil, leads to habitat destruction, pollution, and social conflicts with indigenous communities.

4. **Climate Change**: Changes in climate patterns can lead to increased temperatures, altered rainfall patterns, and more frequent droughts, which can stress the ecosystem and make it more vulnerable to fires and other disturbances.

5. **Fires**: Often set intent

Turn 2's answer correctly resolves "it" to the Amazon rainforest and answers about
deforestation - only possible because `RunnableWithMessageHistory` automatically
replayed turn 1's messages into the prompt before turn 2 ran. Swap `session_id` and
you get a completely independent conversation with no shared memory, which is exactly
how you'd isolate different users/chats in a real application.

## Recap

You've now seen, on the exact same toy data, a complete RAG system end to end:
**document loading** turns real-world sources into a common format, **chunking**
decides what a retrievable unit looks like, **embeddings** turn text into comparable
vectors, **vector databases** store and search those vectors at scale, **search
strategies** (semantic/BM25/hybrid/metadata filtering) decide what gets retrieved,
**reranking** and **MMR** refine that selection for precision and diversity,
**structured output** turns LLM responses into validated, typed data, **LCEL**
composes every piece into one real `retrieve -> prompt -> generate` pipeline, and
**memory** lets that pipeline hold a multi-turn conversation instead of answering
each question in isolation.
